# 2.5 Tool use: tool functions, schemas, message blocks, tool results

Step 2, lesson 5 of [Building with the Claude API](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287747) (Anthropic Academy).
Exam domains: **2 Tool Design & MCP Integration** (tool descriptions, input validation, error responses) and **1 Agentic Architecture & Orchestration** (the request → tool call → result cycle that every agent is built on).

What this notebook covers, in the course's order:

1. What tool use is, and the project: a reminders assistant
2. Tool functions, with input validation and clear errors
3. Tool schemas: name, description, input schema
4. A request with tools, and the blocks that come back
5. Sending a tool result and getting the final answer
6. When a tool fails: `is_error`

> **Model used here:** `claude-sonnet-5-5`. **Cost:** 3 requests.
>
> Docs: [Tool use overview](https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview), [Implement tool use](https://platform.claude.com/docs/en/agents-and-tools/tool-use/implement-tool-use).

In [1]:
import json
from datetime import datetime, timedelta

from anthropic import Anthropic
from anthropic.types import Message, ToolParam
from dateutil.relativedelta import relativedelta
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()
model = "claude-sonnet-5-5"


def text_of(message):
    """Return the text of the first text block in a response ("" if there is none)."""
    return next((block.text for block in message.content if block.type == "text"), "")

## 1. What tool use is

Claude only knows what was in its training data and what you put in the prompt. It doesn't know today's date, can't look up your orders and can't set a reminder. **Tool use** lets it ask *your code* to do those things:

1. You send the user's message **plus a list of tools** (name, description, input schema).
2. If Claude decides a tool would help, it stops and replies with a **`tool_use` block**: which tool, with which inputs. `stop_reason` is `"tool_use"`.
3. **Your code** runs the function. Claude never runs anything itself.
4. You send the output back as a **`tool_result` block**.
5. Claude uses it to write the answer (or asks for another tool, lesson 2.6).

**The project** (the course's): a reminders assistant. Ask it "remind me about my dentist appointment in 3 weeks" and it needs three tools:

| Tool | Why Claude needs it |
|---|---|
| `get_current_datetime` | it doesn't know what time it is |
| `add_duration_to_datetime` | date arithmetic ("177 days from now") is easy to get wrong in its head |
| `set_reminder` | it has no way to store anything |

## 2. Tool functions

A tool function is an ordinary Python function. Two habits from the course make it work well with Claude:

- **Validate the inputs** and raise an error with a **clear message** when they're wrong. The message goes back to Claude, which can then correct its call ("unit must be one of ...") instead of guessing.
- Use **descriptive names** for the function and its arguments; they end up in the schema Claude reads.

`set_reminder` doesn't really schedule anything: it stores the reminder in a list, which is enough to see what Claude asked for.

In [2]:
UNITS = ("seconds", "minutes", "hours", "days", "weeks", "months", "years")
REMINDERS = []


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


def add_duration_to_datetime(datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"):
    if unit not in UNITS:
        raise ValueError(f"unit must be one of {', '.join(UNITS)}; got {unit!r}")
    try:
        start = datetime.strptime(datetime_str, input_format)
    except ValueError:
        raise ValueError(f"datetime_str {datetime_str!r} does not match input_format {input_format!r}")
    if unit in ("months", "years"):
        if duration != int(duration):
            raise ValueError("months and years need a whole number")
        result = start + relativedelta(**{unit: int(duration)})
    else:
        result = start + timedelta(**{unit: duration})
    return result.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    if not content.strip():
        raise ValueError("content cannot be empty")
    try:
        when = datetime.fromisoformat(timestamp)
    except ValueError:
        raise ValueError(f"timestamp must be ISO 8601, e.g. 2050-06-27T09:00:00; got {timestamp!r}")
    REMINDERS.append({"content": content, "timestamp": when.isoformat()})
    return f"Reminder set for {when.isoformat()}: {content}"

Tested locally first, no API needed. Bad inputs give errors that say what to fix:

In [3]:
print(get_current_datetime("%H:%M"))
print(add_duration_to_datetime("2050-01-01", 177, "days"))
print(add_duration_to_datetime("2050-01-31", 1, "months"))

for bad_call in (
    lambda: add_duration_to_datetime("2050-01-01", 3, "fortnights"),
    lambda: add_duration_to_datetime("01/01/2050", 3, "days"),
    lambda: set_reminder("Dentist", "next tuesday"),
):
    try:
        bad_call()
    except ValueError as e:
        print("ValueError:", e)

22:17
Monday, June 27, 2050 12:00:00 AM
Monday, February 28, 2050 12:00:00 AM
ValueError: unit must be one of seconds, minutes, hours, days, weeks, months, years; got 'fortnights'
ValueError: datetime_str '01/01/2050' does not match input_format '%Y-%m-%d'
ValueError: timestamp must be ISO 8601, e.g. 2050-06-27T09:00:00; got 'next tuesday'


## 3. Tool schemas

Claude never sees your Python code. It sees a **schema** for each tool:

- `name`: how Claude refers to the tool;
- `description`: what it does, **when to use it**, what it returns and its limits. Claude chooses tools mostly from this text, so 3–4 full sentences beat a one-liner;
- `input_schema`: a JSON Schema for the arguments, each with its own description, and `required` for the ones without a default.

The course suggests asking Claude to draft a schema from the function's code, then checking it. `ToolParam` is only a type hint for the dict.

In [4]:
get_current_datetime_schema: ToolParam = {
    "name": "get_current_datetime",
    "description": (
        "Returns the current local date and time as a string. "
        "Use it whenever the user refers to 'now', 'today', 'tomorrow' or any time relative to the present, "
        "because you do not know the current date. The output format is controlled by date_format."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A Python strftime format string, e.g. '%Y-%m-%d %H:%M:%S' (the default) or '%H:%M'.",
            }
        },
        "required": [],
    },
}

add_duration_to_datetime_schema: ToolParam = {
    "name": "add_duration_to_datetime",
    "description": (
        "Adds a duration to a date or datetime and returns the result, formatted like "
        "'Thursday, June 27, 2050 12:00:00 AM'. Use it for any date arithmetic (e.g. '177 days after March 3') "
        "instead of calculating dates yourself. Months and years take whole numbers and respect month lengths."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {"type": "string", "description": "The starting date or datetime, matching input_format."},
            "duration": {"type": "number", "description": "How many units to add. Use a negative number to subtract."},
            "unit": {"type": "string", "enum": list(UNITS), "description": "The unit of duration."},
            "input_format": {
                "type": "string",
                "description": "Python strptime format of datetime_str. Default '%Y-%m-%d'; use '%Y-%m-%d %H:%M:%S' if it has a time.",
            },
        },
        "required": ["datetime_str", "duration", "unit"],
    },
}

set_reminder_schema: ToolParam = {
    "name": "set_reminder",
    "description": (
        "Creates a reminder that will notify the user at the given time. "
        "Use it only after you know the exact date and time; work it out with the other tools first. "
        "Returns a confirmation with the stored timestamp."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {"type": "string", "description": "What to remind the user about, e.g. 'Dentist appointment at 10:00'."},
            "timestamp": {"type": "string", "description": "When to send the reminder, in ISO 8601: 'YYYY-MM-DDTHH:MM:SS'."},
        },
        "required": ["content", "timestamp"],
    },
}

## 4. A request with tools, and the blocks that come back

The course's helpers from 2.1 change a little now:

- `chat` takes `tools` and returns the **whole `Message`**, not just its text, because the reply may contain tool calls;
- `add_assistant_message` accepts a `Message` and stores its **full list of content blocks**.

Why the full list? The reply can hold several blocks: possibly a `thinking` block, some `text`, and one or more `tool_use` blocks. The next request must contain that exact assistant turn, because each `tool_result` you send points at a `tool_use` id in it. Keeping only the text breaks the link (and the API returns an error).

In [5]:
def add_user_message(messages, content):
    """content: a string, or a list of blocks (e.g. tool_result blocks)."""
    messages.append({"role": "user", "content": content})


def add_assistant_message(messages, content):
    """content: a string, or a Message (its full list of blocks is kept)."""
    if isinstance(content, Message):
        content = content.content
    messages.append({"role": "assistant", "content": content})


def chat(messages, system=None, tools=None, max_tokens=2000):
    params = {"model": model, "max_tokens": max_tokens, "messages": messages}
    if system:
        params["system"] = system
    if tools:
        params["tools"] = tools
    return client.messages.create(**params)

In [6]:
messages = []
add_user_message(messages, "What is the exact time right now, formatted as HH:MM:SS?")
response = chat(messages, tools=[get_current_datetime_schema])

print("stop_reason:", response.stop_reason)
for block in response.content:
    if block.type == "tool_use":
        print(f"tool_use  id={block.id}  name={block.name}  input={block.input}")
    elif block.type == "text":
        print("text     ", repr(block.text))
    else:
        print(block.type)

stop_reason: tool_use
tool_use  id=toolu_011ahZZKQZkaKqoAAmgndNPm  name=get_current_datetime  input={'date_format': '%H:%M:%S'}


Claude didn't answer: it asked for `get_current_datetime`, and chose the `date_format` argument itself from the user's wording. `block.input` is already a Python dict, so it can be passed straight to the function.

## 5. Sending the tool result

Run the function, then send a **user** message whose content is a list with one `tool_result` block:

- `tool_use_id`: the `id` of the `tool_use` block it answers;
- `content`: the output, as a string (or a list of text/image blocks);
- `is_error`: `False` here (section 6 shows `True`).

The follow-up request must include the same `tools`, and the whole conversation so far.

In [7]:
add_assistant_message(messages, response)          # the full assistant turn, tool_use block included

tool_use = next(block for block in response.content if block.type == "tool_use")
result = get_current_datetime(**tool_use.input)
print("tool output:", result)

add_user_message(messages, [{
    "type": "tool_result",
    "tool_use_id": tool_use.id,
    "content": result,
    "is_error": False,
}])

final = chat(messages, tools=[get_current_datetime_schema])
print("stop_reason:", final.stop_reason)
print("Claude:", text_of(final))

tool output: 22:17:52


stop_reason: end_turn
Claude: The current time is **22:17:52**.


In [8]:
# The conversation now has four messages: question, tool call, tool result, answer
add_assistant_message(messages, final)
for m in messages:
    content = m["content"]
    kinds = [content] if isinstance(content, str) else [b["type"] if isinstance(b, dict) else b.type for b in content]
    print(f"{m['role']:<10}{kinds}")

user      ['What is the exact time right now, formatted as HH:MM:SS?']
assistant ['tool_use']
user      ['tool_result']
assistant ['text']


## 6. When a tool fails: `is_error`

Tools fail: a service times out, an input is invalid, the user isn't allowed to do something. Don't hide the failure, and don't return an empty string (which looks like "no data"). Send a `tool_result` with **`is_error: True`** and a message Claude can act on.

> **Beyond the course.** The exam guide recommends *structured* error content: what kind of error it is and whether a retry could help, so the agent can choose between retrying, trying something else, or explaining the problem to the user. Here the error is simulated: the same tool call as above is answered with a permission error instead of the time.

In [9]:
error_messages = messages[:2]  # the question and Claude's tool_use turn, from above

add_user_message(error_messages, [{
    "type": "tool_result",
    "tool_use_id": tool_use.id,
    "content": json.dumps({
        "error_category": "permission",   # e.g. transient | validation | permission | business_rule
        "is_retryable": False,
        "message": "This assistant is not allowed to read the system clock on this device.",
    }),
    "is_error": True,
}])

reply = chat(error_messages, tools=[get_current_datetime_schema])
print("stop_reason:", reply.stop_reason)
print("Claude:", text_of(reply) or [b.type for b in reply.content])

stop_reason: end_turn
Claude: I can't give you the exact time. The clock tool returned a permission error saying this assistant isn't allowed to read the system clock on this device. Retrying won't help, and I have no other source for the current time.

Your device's clock will show it. If you tell me the time, I can help with calculations or conversions from there.


**Exam angle.**

- **Domain 2, descriptions:** Claude picks tools from their descriptions. When it calls the wrong tool or skips one, the first fix is a clearer description (what it does, when to use it, what it returns, how it differs from similar tools), before adding routing logic or more tools.
- **Domain 2, errors:** return failures as `tool_result` with `is_error: true` and a useful message: the category, whether retrying makes sense, what to fix. Keep "the lookup failed" separate from "the lookup worked and found nothing": the second is a normal result, not an error.
- **Domain 2, validation:** check inputs in the tool and say exactly what was wrong; the model corrects its next call from that message.
- **Domain 1:** the assistant turn goes back into the history **complete** (all blocks), and each `tool_result` matches a `tool_use_id`. Lesson 2.6 turns this into a loop.

## Recap

- Tools = schema (name, description, JSON Schema) + your function. Claude asks; your code runs.
- A tool call arrives as a `tool_use` block with `stop_reason == "tool_use"`; `block.input` is a dict.
- Append the full `response.content` as the assistant turn; answer with a `tool_result` block that has the matching `tool_use_id`.
- Failures go back as `tool_result` with `is_error: True` and a clear, structured message.

Next: [2.6 Tool loops, multiple tools and built-in tools](06-tool-use-loops-and-built-in-tools.ipynb).